A character-level RNN reads words as a series of characters - outputting a prediction and “hidden state” at each step, feeding its previous hidden state into each next step. We take the final prediction to be the output, i.e. which class the word belongs

Specifically, we’ll train on a few thousand surnames from 18 languages of origin, and predict which language a name is from based on the spelling

In [20]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Setting the Default Device to CUDA for the rest of the Project if its available
torch.set_default_device(device)

# checking which device is current being used
print(torch.get_default_device())

cuda:0


# Preparing the Data

The first step is to define and clean our data. Initially, we need to convert Unicode to plain ASCII to limit the RNN input layers. This is accomplished by converting Unicode strings to ASCII and allowing only a small set of allowed characters.

In [21]:
import string
import unicodedata

# We can use "_" to represent an out-of-vocabulary character, that is, any character we are not handling in our model
allowed_chars = string.ascii_letters+ ".,;'"+"_"
n_letters = len(allowed_chars)

# Turn a Unicode string to plain ASCII
def unicodeToAscii(s):
  return ''.join(
      c for c in unicodedata.normalize('NFD', s)  # This function separates letters from their accents or diacritics (The marks that we usually see on letters with which the accent can be changed)
      # (i.e, café will be converted to cafe + the char above e)
      # Normalization Form Canonical Decomposition(NFD)
      if unicodedata.category(c) != 'Mn' #(Mark, Non Spacing)
        and c in allowed_chars
  )

# Example
print(unicodeToAscii('Ślusàrski'))
print(n_letters)
print(allowed_chars)

Slusarski
57
abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ.,;'_


# Turning the Names into Tensors


Now that we have all the names organized, we need to turn them into Tensors to make any use of them

To represent a single letter, we use a “one-hot vector” of size <1 x n_letters(i.e. chars in your vocabulary>. A one-hot vector is filled with 0s except for a 1 at index of the current letter, e.g. "b" = <0 1 0 0 0 ...>.

To make a word we join a bunch of those into a 2D matrix <no_of_letters_in_specific_word x 1 x n_letters>

That extra 1 dimension is because PyTorch assumes everything is in batches - we’re just using a batch size of 1 here.

In [22]:
# Find letter index from allowed_chars e.g. "a" = 0
def letter_to_index(letter):
  # return our out-of-vocabulary character (i.e. we are using _ if any char is out of the vocab we are using ) if we encounter a letter unknown to our model
  if letter not in allowed_chars:
    return allowed_chars.find("_")
  else:
    return allowed_chars.find(letter)

# Turning the word into <no_of_letters_in_a_word x 1 x n_letters(your vocabulary size(i.e. allowed_chars))>
def line_to_tensor(line):
  tensor = torch.zeros(len(line), 1, n_letters)
  for li, letter in enumerate(line):
    tensor[li][0][letter_to_index(letter)] = 1
  return tensor

print(line_to_tensor('Mujeeb'))


tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0

# Congratulations, you have built the foundational tensor objects for this learning task! You can use a similar approach for other RNN tasks with text.

we will use the Dataset and DataLoader classes to hold our dataset. Each Dataset needs to implement three functions: __init__, __len__, and __getitem__.

In [27]:
from io import open
import glob
import os, time
import torch
from torch.utils.data import Dataset


class NamesDataset(Dataset):
  def __init__(self, data_dir):
    self.data_dir = data_dir # path of the data directory
    self.load_time = time.localtime # Time when the data was loaded
    labels_set = set() # set of all classes
    self.data = []
    self.data_tensors = []
    self.labels =[]
    self.labels_tensors = []

    #read all the ``.txt`` files in the specified directory
    text_files = glob.glob(os.path.join(data_dir, "*.txt"))
    for filename in text_files:
      label = os.path.splitext(os.path.basename(filename))[0] # The os.path.basename returns the last file in the route like resume.txt will be
      # fetched from mujeeb/data/resume.txt and os.path.splitext returns the file name and extension mujeeb.txt => ('mujeeb', '.txt')
      labels_set.add(label) # add the label to the set for the uniqueness
      lines = open(filename, encoding='utf-8').read().strip().split("\n")
      for name in lines:
        self.data.append(name)
        self.data_tensors.append(line_to_tensor(name))
        self.labels.append(label)

    # cache the tensor representation of labels
    self.labels_uniq = list(labels_set)
    for idx in range(len(self.labels)):
      temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
      self.labels_tensors.append(temp_tensor)

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    data_item = self.data[idx]
    data_label = self.labels[idx]
    data_tensor = self.data_tensors[idx]
    label_tensor = self.labels_tensors[idx]
    return label_tensor, data_tensor, data_label, data_item


In [24]:
all_data = NamesDataset('./data/names')
print(len(all_data))
print(all_data[0])

20074
(tensor([0], device='cuda:0'), tensor([[[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
          0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.

Dataset object allows us to easily split the data into train and test sets. Here we create a 85/15

**torch.utils.data** has more useful utilities. Here we specify a generator since we need to use the

same device as PyTorch defaults to above

In [28]:
train_set, test_set = torch.utils.data.random_split(all_data, [.85, .15], generator=torch.Generator(device=device).manual_seed(2024))

print(f"Train set size: {len(train_set)}")
print(f"Test set size: {len(test_set)}")

Train set size: 17063
Test set size: 3011


# Now we have a basic dataset containing 20074 examples where each example is a pairing of label and name. We have also split the dataset into training and testing so we can validate the model that we build.

# Creating the Network